# imprecision-bench: Benchmark Orientation Guide

This notebook introduces the **imprecision-bench** dataset and shows what current language and vision-language models do on it. The benchmark tests a simple pragmatic question: does a model adjust *how precisely* it reports a time depending on social context — saying "8:30" to a police officer but "around half past eight" to a neighbor? Humans do this naturally. Current models largely don't.

**Two tasks:**
- **Task 1 (production):** given a clock image or text description + scenario, complete "It happened ___."  
- **Task 2 (motive elicitation):** given the production + context, explain why that wording was chosen.

**Dataset:** 475 human productions × 12 clock states × 2 contexts (police / neighbor)  
**Source:** Mühlenbernd & Solt (2022), *Linguistics Vanguard*, DOI: 10.1515/lingvan-2022-0035

## 1. Setup

In [ ]:
from datasets import load_dataset
import pandas as pd
from IPython.display import display, Image as IPImage
import io

HF_REPO_ID = "RolandM/imprecision-bench"
RESULTS_CSV = "results.csv"

# Models included in this notebook
MODELS = {
    "gpt_4o_mini":             "GPT-4o mini (OpenAI)",
    "claude_haiku_4_5":        "Claude Haiku 4.5 (Anthropic)",
    "google_gemini_2_5_flash": "Gemini 2.5 Flash (Google)",
}

### API keys

To re-run or extend the evaluation, copy `.env.example` to `.env` and fill in your keys:

```
OPENAI_API_KEY=...       # for gpt-4o, gpt-4o-mini
ANTHROPIC_API_KEY=...    # for claude-* models
OPENROUTER_API_KEY=...   # for google/gemini-* via OpenRouter
TOGETHER_API_KEY=...     # for open-weight models via Together.ai
```

Then run:
```bash
python evaluate.py --model gpt-4o-mini --rows 50   # pilot: first 50 rows
python evaluate.py --model gpt-4o-mini --full       # full: all 475 rows
```

## 2. The Benchmark Data

In [ ]:
ds = load_dataset(HF_REPO_ID, split="train")
print(f"Dataset: {len(ds)} rows")

context_names  = ds.features["context"].names
stimulus_names = ds.features["stimulus_type"].names

df_full = ds.to_pandas()
df_full["context_str"]  = df_full["context"].map(lambda x: context_names[x])
df_full["stimulus_str"] = df_full["stimulus_type"].map(lambda x: stimulus_names[x])

print("\nCondition breakdown:")
print(df_full.groupby(["target_time", "context_str"]).size().unstack())

In [ ]:
# Show a sample row: clock image, description, and human response
sample = df_full[df_full["context_str"] == "police"].iloc[0]

print(f"Target time     : {sample['target_time']}")
print(f"Context         : {context_names[sample['context']]}")
print(f"Stimulus type   : {stimulus_names[sample['stimulus_type']]}")
print(f"Clock description: {sample['clock_description']}")
print(f"Prompt          : {sample['prompt']}")
print(f"Human production: {sample['production']}")
print()
print("Clock image:")
img_buf = io.BytesIO()
sample["clock_image"].save(img_buf, format="PNG")
display(IPImage(data=img_buf.getvalue(), width=200))

## 3. Results Overview

Pre-computed results are stored in `results.csv`. Each model adds three columns:
- `{model}_task1_image` — production from clock *image*
- `{model}_task1_text` — production from clock *text description*
- `{model}_task2` — motive explanation

In [ ]:
results = pd.read_csv(RESULTS_CSV)
print(f"Results file: {len(results)} rows, {len(results.columns)} columns")

# Coverage: how many rows each model has answered
coverage = {}
for key, label in MODELS.items():
    col = f"{key}_task1_text"
    if col in results.columns:
        coverage[label] = results[col].notna().sum()

print("\nModel coverage (Task 1 text, non-null rows):")
for label, n in coverage.items():
    print(f"  {label}: {n} / {len(results)}")

## 4. Task 1 — Clock Reading: Image vs. Text

The benchmark includes two versions of Task 1: one where the model sees the **clock image**, and one where it sees a **textual description** of the same clock. This lets us separate visual grounding ability from linguistic pragmatics.

In [ ]:
# Show raw outputs side by side for the first 10 rows with data
sample_rows = results.dropna(subset=["gpt_4o_mini_task1_image"]).head(10)

display_cols = ["target_time", "context"]
for key in MODELS:
    img_col  = f"{key}_task1_image"
    text_col = f"{key}_task1_text"
    if img_col in results.columns:
        display_cols += [img_col, text_col]

pd.set_option("display.max_colwidth", 35)
display(sample_rows[display_cols].reset_index(drop=True))

In [ ]:
# Modality gap: heuristic — does the response contain the correct hour digit?
# A rough proxy for clock-reading accuracy on the image task.
def contains_correct_hour(response, target_time):
    """Check if the response mentions the correct hour (8 for all conditions here)."""
    if pd.isna(response):
        return None
    return "8" in str(response) or "eight" in str(response).lower()

pilot = results.dropna(subset=["gpt_4o_mini_task1_image"]).copy()

print("Correct hour mentioned (image task vs text task):")
print(f"{'Model':<35} {'Image':>8} {'Text':>8}")
print("-" * 55)
for key, label in MODELS.items():
    img_col  = f"{key}_task1_image"
    text_col = f"{key}_task1_text"
    if img_col not in pilot.columns:
        continue
    img_acc  = pilot[img_col].apply(lambda r: contains_correct_hour(r, None)).mean()
    text_acc = pilot[text_col].apply(lambda r: contains_correct_hour(r, None)).mean()
    print(f"{label:<35} {img_acc:>7.0%} {text_acc:>7.0%}")

## 5. Task 1 — Pragmatic Shift: Police vs. Neighbor

The core question: does a model produce *more precise* times for a police officer than for a neighbor? Humans do — they round in casual contexts and give exact times in formal ones.

In [ ]:
# Simple proxy for precision: does the response contain ":" (digital time like "8:30")?
# Imprecise responses tend to use words ("half past eight", "around eight").
def is_digital(response):
    if pd.isna(response):
        return None
    return ":" in str(response)

pilot = results.dropna(subset=["gpt_4o_mini_task1_text"]).copy()

print("Rate of digital time format (e.g. \"8:30\") — text task:")
print(f"{'Model':<35} {'Police':>8} {'Neighbor':>8} {'Shift':>8}")
print("-" * 65)
for key, label in MODELS.items():
    col = f"{key}_task1_text"
    if col not in pilot.columns:
        continue
    pilot["digital"] = pilot[col].apply(is_digital)
    police   = pilot[pilot["context"] == "police"]["digital"].mean()
    neighbor = pilot[pilot["context"] == "neighbor"]["digital"].mean()
    shift    = police - neighbor
    print(f"{label:<35} {police:>7.0%} {neighbor:>7.0%} {shift:>+7.0%}")

# Human baseline
df_full["digital"] = df_full["production"].apply(lambda r: is_digital(r))
h_police   = df_full[df_full["context_str"] == "police"]["digital"].mean()
h_neighbor = df_full[df_full["context_str"] == "neighbor"]["digital"].mean()
print(f"{'Human baseline':<35} {h_police:>7.0%} {h_neighbor:>7.0%} {h_police - h_neighbor:>+7.0%}")

## 6. Task 2 — Motive Elicitation (Sample)

Task 2 asks the model to explain *why* a given time expression was chosen. This is not yet scored quantitatively — it is an open contribution opportunity.

In [ ]:
# Show three Task 2 examples for each model
sample = results.dropna(subset=["gpt_4o_mini_task2"]).head(3)
for _, row in sample.iterrows():
    print(f"[{row['context']} | {row['target_time']}]")
    print(f"  Human:   {row['human_production']}")
    for key, label in MODELS.items():
        col = f"{key}_task2"
        if col in row and pd.notna(row[col]):
            print(f"  {label[:25]}: {str(row[col])[:120]}")
    print()

## 7. Full Analysis

> **TODO:** This section will contain the full quantitative analysis once all models have been run on the complete 475-row dataset.

Planned metrics:
- Clock-reading accuracy (image vs. text) per model — exact time match against `target_time`
- Pragmatic shift score — Wasserstein distance between police and neighbor production distributions
- Comparison against human baseline Wasserstein distance
- Per-condition breakdown (precise vs. imprecise clock states)

## 8. Open Challenges

**Modality gap — analog clock reading.** All three models read clocks more accurately from text descriptions than from images. GPT-4o mini and Claude Haiku frequently default to "eight o'clock" regardless of what the clock shows. Gemini 2.5 Flash is notably better but still makes errors on off-round times. *Open challenge: better visual grounding for analog clock reading in VLMs.*

**Pragmatic precision calibration.** Models produce digital times at roughly the same rate for police and neighbor contexts. Humans clearly round in casual contexts ("around half past eight") and give exact times in formal ones ("8:28"). Models appear to apply a fixed precision strategy independent of audience. *Open challenge: context-sensitive precision calibration.*

**Task 2 — motive elicitation.** The motive explanations generated by models are fluent and plausible, but they are not yet scored against the human motive annotations. Developing a scoring scheme for pragmatic motive attribution is an open contribution opportunity.

**Open-weight VLMs.** As of mid-2026, benchmarking open-weight VLMs via hosted APIs is non-trivial: most capable models either require dedicated endpoints or have reasoning/thinking modes that interfere with short-response tasks. A local inference setup (e.g. `ollama`, `vllm`) is recommended for open-weight evaluation.

## 9. Extending the Benchmark

**Run the full dataset:**
```bash
python evaluate.py --model gpt-4o-mini --full
```

**Add a new model (OpenAI-compatible):**
```bash
python evaluate.py --model google/gemini-2.5-pro --rows 50  # pilot
python evaluate.py --model google/gemini-2.5-pro --full     # full run
```

**Add a new model (Anthropic):**
```bash
python evaluate.py --model claude-sonnet-4-6 --full
```

Results are merged into `results.csv` automatically — new model columns are added without overwriting existing ones.

**Compute Wasserstein distance** against the human baseline:
```python
from scipy.stats import wasserstein_distance
# human_production_code encodes precision: 0=exact, higher=more imprecise
police   = df[df.context=="police"]["human_production_code"].astype(float)
neighbor = df[df.context=="neighbor"]["human_production_code"].astype(float)
print(wasserstein_distance(police, neighbor))  # human baseline: ~0.27
```